In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np
import pickle
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train_df = pd.read_parquet(PROCESSED_DIR / "train.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "test.parquet")

with open(PROCESSED_DIR / "encoders.pkl", "rb") as f:
    encoders = pickle.load(f)

n_users = encoders["n_users"]
n_movies = encoders["n_movies"]

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

global_mean = train_df["rating"].mean()

print(f"Train: {train_df.shape}, Test: {test_df.shape}")

Train: (802553, 6), Test: (197656, 6)


In [2]:
train_matrix = csr_matrix(
    (train_df["rating"], (train_df["user_idx"], train_df["movie_idx"])),
    shape=(n_users, n_movies)
)

# Item-item similarity: similarity between movie columns
# transpose so rows = movies, cols = users
item_similarity = cosine_similarity(train_matrix.T, dense_output=False)

print(f"Item similarity matrix shape: {item_similarity.shape}")
print(f"Type: {type(item_similarity)}")

Item similarity matrix shape: (3706, 3706)
Type: <class 'scipy.sparse._csr.csr_matrix'>


In [3]:
# Build a fast lookup: user_idx -> {movie_idx: rating} from train
from collections import defaultdict

user_ratings_dict = defaultdict(dict)
for row in train_df.itertuples(index=False):
    user_ratings_dict[row.user_idx][row.movie_idx] = row.rating

item_similarity_csr = item_similarity.tocsr()

def predict_rating_item_cf(user_idx, movie_idx, k=20):
    """Predict rating using top-k similar movies the user has rated."""
    user_rated = user_ratings_dict.get(user_idx, {})
    if not user_rated:
        return global_mean

    # Get similarities of target movie to all movies
    sim_row = item_similarity_csr[movie_idx]
    sim_dict = dict(zip(sim_row.indices, sim_row.data))

    # Intersect with movies this user has rated
    sims_and_ratings = [
        (sim_dict[m], r) for m, r in user_rated.items() if m in sim_dict and sim_dict[m] > 0
    ]

    if not sims_and_ratings:
        return global_mean

    # Take top-k by similarity
    sims_and_ratings.sort(key=lambda x: -x[0])
    sims_and_ratings = sims_and_ratings[:k]

    sims = np.array([s for s, r in sims_and_ratings])
    rats = np.array([r for s, r in sims_and_ratings])

    if sims.sum() == 0:
        return global_mean

    return np.dot(sims, rats) / sims.sum()

# Test on a small sample first (full test set would be slow in pure Python loop)
sample_test = test_df.sample(2000, random_state=42)

preds = [
    predict_rating_item_cf(row.user_idx, row.movie_idx)
    for row in sample_test.itertuples(index=False)
]

sample_test = sample_test.copy()
sample_test["pred_item_cf"] = preds

rmse_item_cf = rmse(sample_test["rating"], sample_test["pred_item_cf"])
mae_item_cf = mae(sample_test["rating"], sample_test["pred_item_cf"])

print(f"Item-Item CF (sample of 2000) -> RMSE: {rmse_item_cf:.4f}, MAE: {mae_item_cf:.4f}")

Item-Item CF (sample of 2000) -> RMSE: 1.0213, MAE: 0.7941


In [4]:
sample_test_large = test_df.sample(10000, random_state=42)

preds_large = [
    predict_rating_item_cf(row.user_idx, row.movie_idx)
    for row in sample_test_large.itertuples(index=False)
]

sample_test_large = sample_test_large.copy()
sample_test_large["pred_item_cf"] = preds_large

rmse_item_cf_large = rmse(sample_test_large["rating"], sample_test_large["pred_item_cf"])
mae_item_cf_large = mae(sample_test_large["rating"], sample_test_large["pred_item_cf"])

print(f"Item-Item CF (sample of 10000) -> RMSE: {rmse_item_cf_large:.4f}, MAE: {mae_item_cf_large:.4f}")

Item-Item CF (sample of 10000) -> RMSE: 0.9937, MAE: 0.7693


In [5]:
import pickle

MODELS_DIR = PROJECT_ROOT / "models"

with open(MODELS_DIR / "item_similarity.pkl", "wb") as f:
    pickle.dump(item_similarity_csr, f)

with open(MODELS_DIR / "item_cf_results.pkl", "wb") as f:
    pickle.dump({
        "rmse": rmse_item_cf_large,
        "mae": mae_item_cf_large,
        "sample_size": 10000,
    }, f)

print("Saved item_similarity.pkl and item_cf_results.pkl to models/")

Saved item_similarity.pkl and item_cf_results.pkl to models/
